# LeetCode 1945 — Sum of Digits of String After Convert bằng Python

**Link đề:** https://leetcode.com/problems/sum-of-digits-of-string-after-convert/
**Độ khó:** Easy · **Chủ đề:** String, Simulation

```python
class Solution:
    def getLucky(self, s: str, k: int) -> int:
        ...
```

## Mục tiêu học
1. Đổi chữ cái thành vị trí trong bảng chữ (`'a' → 1`, …, `'z' → 26`) bằng `ord`.
2. Tính **tổng chữ số** của một số bằng hai cách: qua chuỗi (`str`) và bằng toán (`% 10`, `// 10`).
3. Mô phỏng (simulation) đúng **k lần** biến đổi bằng vòng lặp.
4. Rút ra insight: lần biến đổi đầu có thể làm **thẳng từ các chữ cái**, không cần dựng số khổng lồ.
5. Trace, chứng minh, Time/Space, các lỗi hay gặp và biến thể (digital root).

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — đổi chữ thành số rồi cộng chữ số k lần

Cho chuỗi chữ thường `s` và số nguyên `k`.
1. **Convert:** thay mỗi chữ bằng vị trí của nó trong bảng chữ (`a=1, b=2, …, z=26`), ghép lại thành một số nguyên.
2. **Transform:** thay số đó bằng **tổng các chữ số** của nó.
3. Lặp bước Transform **tổng cộng `k` lần**. Trả về số cuối cùng.

Ví dụ `s = "zbax"`, `k = 2`:
- Convert: `"zbax" → (26)(2)(1)(24) → "262124" → 262124`
- Transform #1: `2+6+2+1+2+4 = 17`
- Transform #2: `1+7 = 8` → kết quả `8`.

| `s` | `k` | Output |
|---|:---:|:---:|
| `"iiii"` | 1 | `36` (9999 → 36) |
| `"leetcode"` | 2 | `6` (12552031545 → 33 → 6) |
| `"zbax"` | 2 | `8` |

**Constraints:** `1 <= s.length <= 100`, `1 <= k <= 10`, `s` chỉ gồm chữ thường.

Lưu ý: dù kết quả đã là một chữ số, ta **vẫn** phải biến đổi đủ `k` lần (biến đổi một số có 1 chữ số thì giữ nguyên nó, nên không ảnh hưởng).

In [1]:
# Kiểm chứng ví dụ "zbax" bằng tay
print(2 + 6 + 2 + 1 + 2 + 4)   # 17
print(1 + 7)                   # 8

17
8


## 2. Nền tảng Python cần dùng

### 2.1. `ord(ch)` — mã số của ký tự; vị trí trong bảng chữ

`ord('a') = 97`, `ord('b') = 98`, … Các chữ thường có mã **liên tiếp**, nên vị trí 1-based là `ord(ch) - ord('a') + 1`.

In [2]:
for ch in "abxyz":
    print(ch, ord(ch), "-> vị trí", ord(ch) - ord("a") + 1)

a 97 -> vị trí 1
b 98 -> vị trí 2
x 120 -> vị trí 24
y 121 -> vị trí 25
z 122 -> vị trí 26


### 2.2. `str(n)`, `int(chuoi)` và `"".join(...)`

- `str(26)` → `"26"`; `int("262124")` → `262124`.
- `"".join(list_chuoi)` ghép các chuỗi lại, không chèn gì ở giữa.

In [3]:
parts = [str(ord(ch) - ord("a") + 1) for ch in "zbax"]
print(parts)                  # ['26', '2', '1', '24']
joined = "".join(parts)
print(joined, int(joined))    # 262124 262124

['26', '2', '1', '24']
262124 262124


### 2.3. Tổng chữ số — cách chuỗi

Duyệt từng ký tự của `str(n)`, đổi về `int` rồi cộng.

In [4]:
n = 262124
print([int(d) for d in str(n)])        # [2, 6, 2, 1, 2, 4]
print(sum(int(d) for d in str(n)))     # 17

[2, 6, 2, 1, 2, 4]
17


### 2.4. Tổng chữ số — cách toán học `% 10` và `// 10`

- `n % 10` lấy chữ số **cuối**; `n // 10` **bỏ** chữ số cuối.
- Lặp tới khi `n == 0`.

Ví dụ `n = 17`: `17 % 10 = 7`, `17 // 10 = 1`; `1 % 10 = 1`, `1 // 10 = 0` → tổng `7 + 1 = 8`.

In [5]:
def digit_sum(n):
    total = 0
    while n > 0:
        total += n % 10   # lấy chữ số cuối
        n //= 10          # bỏ chữ số cuối
    return total

print(digit_sum(262124), digit_sum(17), digit_sum(9), digit_sum(0))   # 17 8 9 0
print(123 / 10, 123 // 10, 123 % 10)    # 12.3 12 3 — đừng nhầm / với //

17 8 9 0
12.3 12 3


### 2.5. `divmod` — lấy cả thương và dư một lần

`divmod(n, 10)` trả cặp `(n // 10, n % 10)`. Có thể dùng để viết gọn vòng lặp.

In [6]:
print(divmod(262124, 10))   # (26212, 4)
n, total = 262124, 0
while n:
    n, d = divmod(n, 10)
    total += d
print(total)                 # 17

(26212, 4)
17


## 3. Tính tay — ví dụ `"leetcode"`, `k = 2`

| Chữ | l | e | e | t | c | o | d | e |
|---|---|---|---|---|---|---|---|---|
| Vị trí | 12 | 5 | 5 | 20 | 3 | 15 | 4 | 5 |

Ghép: `"12552031545"`.
Transform #1: `1+2+5+5+2+0+3+1+5+4+5 = 33`.
Transform #2: `3+3 = 6` → đáp án `6`.

Để ý: số sau Convert có tới **11 chữ số**; với `s` dài 100 chữ cái, số có tới **200 chữ số**. Python xử lý được số nguyên lớn tuỳ ý, nhưng nhiều ngôn ngữ khác (C++, Java) thì tràn số.

In [7]:
digits = "".join(str(ord(ch) - 96) for ch in "leetcode")
print(digits, len(digits), "chữ số")
print(" + ".join(digits), "=", sum(int(d) for d in digits))

12552031545 11 chữ số
1 + 2 + 5 + 5 + 2 + 0 + 3 + 1 + 5 + 4 + 5 = 33


## 4. Cách 1 — Mô phỏng đúng nguyên văn đề (baseline)

Làm **đúng từng bước** đề mô tả: dựng chuỗi số → đổi thành `int` → lặp `k` lần "tổng chữ số" bằng cách chuỗi.

In [8]:
def get_lucky_literal(s, k):
    num = int("".join(str(ord(ch) - ord("a") + 1) for ch in s))   # Convert
    for _ in range(k):                                           # k lần Transform
        num = sum(int(d) for d in str(num))
    return num

print(get_lucky_literal("iiii", 1))      # 36
print(get_lucky_literal("leetcode", 2))  # 6
print(get_lucky_literal("zbax", 2))      # 8

36
6
8


### 4.1. `for _ in range(k)` — lặp đúng `k` lần

Dấu gạch dưới `_` là quy ước "biến không dùng tới". `range(k)` sinh `0, 1, …, k-1` → đúng `k` lần lặp.

In [9]:
for _ in range(3):
    print("lặp")
print(list(range(3)))

lặp
lặp
lặp
[0, 1, 2]


### 4.2. Nút thắt của baseline

- Dựng một số nguyên **200 chữ số** chỉ để rồi… tách lại thành chữ số. `int(...)` rồi `str(...)` là hai bước chuyển đổi thừa.
- Ở ngôn ngữ không có số nguyên lớn, cách này **tràn số** ngay.

Câu hỏi: lần Transform đầu tiên chỉ cần **tổng các chữ số** của chuỗi ghép. Tổng đó có tính được thẳng từ từng chữ cái không?

## 5. Bước đột phá — Transform #1 = tổng "tổng chữ số" của từng chữ cái

Chuỗi ghép là nối các chuỗi con `str(vị_trí)`. Tổng chữ số của một chuỗi ghép = tổng (tổng chữ số của từng mảnh). Ví dụ:

`"zbax"` → mảnh `26, 2, 1, 24` → `(2+6) + 2 + 1 + (2+4) = 8 + 2 + 1 + 6 = 17`.

Vị trí `p` chỉ nằm trong `1..26` nên tổng chữ số của nó là `p // 10 + p % 10` (vì `p` có tối đa 2 chữ số).

Sau Transform #1, giá trị ≤ `100 · 10 = 1000` (mỗi chữ cái đóng góp tối đa `1+9 = 10` với chữ `s`(19)), nên `k-1` lần còn lại rất rẻ.

In [10]:
for ch in "zbax":
    p = ord(ch) - 96
    print(ch, "p =", p, "-> tổng chữ số =", p // 10, "+", p % 10, "=", p // 10 + p % 10)
print("Transform #1 =", sum((ord(ch) - 96) // 10 + (ord(ch) - 96) % 10 for ch in "zbax"))

z p = 26 -> tổng chữ số = 2 + 6 = 8
b p = 2 -> tổng chữ số = 0 + 2 = 2
a p = 1 -> tổng chữ số = 0 + 1 = 1
x p = 24 -> tổng chữ số = 2 + 4 = 6
Transform #1 = 17


In [11]:
# Chữ cái nào đóng góp nhiều nhất vào Transform #1?
best = max("abcdefghijklmnopqrstuvwxyz", key=lambda ch: digit_sum(ord(ch) - 96))
print(best, ord(best) - 96, digit_sum(ord(best) - 96))   # 's' = 19 -> 10

s 19 10


`max(..., key=hàm)` chọn phần tử làm `hàm(phần_tử)` lớn nhất. `lambda ch: ...` là cách viết một hàm ngắn không cần đặt tên. Kết quả: chữ `s` (vị trí 19) đóng góp `1 + 9 = 10` — vì vậy Transform #1 không vượt quá `10 · len(s) ≤ 1000`.

## 6. Lời giải chính — giảng từng đoạn rồi ghép

**Đoạn 1 — Transform #1 thẳng từ chữ cái:** với mỗi `ch`, `p = ord(ch) - ord('a') + 1`, cộng `p // 10 + p % 10` vào `num`.

**Đoạn 2 — `k - 1` lần còn lại:** `num = digit_sum(num)`.

**Đoạn 3 — trả `num`.**

In [12]:
def get_lucky(s, k):
    # Đoạn 1: Convert + Transform #1 gộp làm một
    num = 0
    for ch in s:
        p = ord(ch) - ord("a") + 1
        num += p // 10 + p % 10
    # Đoạn 2: k - 1 lần Transform còn lại
    for _ in range(k - 1):
        total = 0
        while num > 0:
            total += num % 10
            num //= 10
        num = total
    return num                                 # Đoạn 3

print(get_lucky("iiii", 1), get_lucky("leetcode", 2), get_lucky("zbax", 2))   # 36 6 8

36 6 8


## 7. TRACE từng bước

Ví dụ `s = "zbax"`, `k = 2`.

**Đoạn 1:**

| ch | p | p // 10 | p % 10 | num sau |
|---|---:|---:|---:|---:|
| z | 26 | 2 | 6 | 8 |
| b | 2 | 0 | 2 | 10 |
| a | 1 | 0 | 1 | 11 |
| x | 24 | 2 | 4 | 17 |

**Đoạn 2** (chạy `k - 1 = 1` lần): `num = 17` → bóc `7` (total 7, num 1) → bóc `1` (total 8, num 0) → `num = 8`.

In [13]:
def trace_get_lucky(s, k):
    print(f"s = {s!r}, k = {k}")
    num = 0
    for ch in s:
        p = ord(ch) - ord("a") + 1
        num += p // 10 + p % 10
        print(f"  ch={ch} p={p:>2} -> cộng {p // 10}+{p % 10}, num = {num}")
    print(f"Sau Transform #1: num = {num}")
    for t in range(2, k + 1):
        total, x = 0, num
        while x > 0:
            total += x % 10
            print(f"    bóc chữ số {x % 10}, total = {total}, còn lại {x // 10}")
            x //= 10
        num = total
        print(f"Sau Transform #{t}: num = {num}")
    return num

print("Kết quả:", trace_get_lucky("zbax", 2))

s = 'zbax', k = 2
  ch=z p=26 -> cộng 2+6, num = 8
  ch=b p= 2 -> cộng 0+2, num = 10
  ch=a p= 1 -> cộng 0+1, num = 11
  ch=x p=24 -> cộng 2+4, num = 17
Sau Transform #1: num = 17
    bóc chữ số 7, total = 7, còn lại 1
    bóc chữ số 1, total = 8, còn lại 0
Sau Transform #2: num = 8
Kết quả: 8


### 7.1. Trace nhánh đặc biệt: `k` lớn hơn số lần cần thiết

`s = "leetcode"`, `k = 4`: sau 2 lần đã ra `6` (một chữ số). Hai lần sau tổng chữ số của `6` vẫn là `6` — giá trị đứng yên. Đây là **điểm bất động** (fixed point) của phép biến đổi.

In [14]:
print("Kết quả:", trace_get_lucky("leetcode", 4))

s = 'leetcode', k = 4
  ch=l p=12 -> cộng 1+2, num = 3
  ch=e p= 5 -> cộng 0+5, num = 8
  ch=e p= 5 -> cộng 0+5, num = 13
  ch=t p=20 -> cộng 2+0, num = 15
  ch=c p= 3 -> cộng 0+3, num = 18
  ch=o p=15 -> cộng 1+5, num = 24
  ch=d p= 4 -> cộng 0+4, num = 28
  ch=e p= 5 -> cộng 0+5, num = 33
Sau Transform #1: num = 33
    bóc chữ số 3, total = 3, còn lại 3
    bóc chữ số 3, total = 6, còn lại 0
Sau Transform #2: num = 6
    bóc chữ số 6, total = 6, còn lại 0
Sau Transform #3: num = 6
    bóc chữ số 6, total = 6, còn lại 0
Sau Transform #4: num = 6
Kết quả: 6


## 8. Các cách giải khác

### 8.1. Cách 3 — dừng sớm khi `num < 10`

Vì số một chữ số là điểm bất động, khi `num < 10` ta có thể `break` mà không đổi kết quả. Với `k ≤ 10` lợi ích không lớn, nhưng đây là tư duy tốt.

In [15]:
def get_lucky_early_stop(s, k):
    num = sum(digit_sum(ord(ch) - 96) for ch in s)
    for _ in range(k - 1):
        if num < 10:          # đã là điểm bất động
            break
        num = digit_sum(num)
    return num

print(get_lucky_early_stop("leetcode", 10), get_lucky_early_stop("zbax", 2))

6 8


### 8.2. Cách 4 — chuỗi chữ số thay vì số nguyên lớn

Ghép chuỗi chữ số nhưng **không** đổi sang `int` — cộng thẳng từng ký tự. Tránh số lớn nhưng vẫn tạo chuỗi dài `O(n)`.

In [16]:
def get_lucky_string(s, k):
    digits = "".join(str(ord(ch) - 96) for ch in s)
    num = sum(int(d) for d in digits)               # Transform #1 trên chuỗi
    for _ in range(k - 1):
        num = sum(int(d) for d in str(num))
    return num

print(get_lucky_string("iiii", 1), get_lucky_string("leetcode", 2), get_lucky_string("zbax", 2))

36 6 8


### 8.3. Ghi chú — digital root (căn số)

Nếu `k` đủ lớn (biến đổi tới khi còn một chữ số), kết quả là **digital root**: với `N > 0`, `dr(N) = 1 + (N - 1) % 9`, vì tổng chữ số giữ nguyên **phần dư khi chia 9** (do `10 ≡ 1 (mod 9)`).

Nhưng bài này cố định `k` lần, có thể dừng **trước** khi còn một chữ số (ví dụ `"iiii"`, `k=1` ra `36`). Vì vậy digital root **không** thay được mô phỏng — chỉ dùng để kiểm tra chéo khi `k` lớn.

In [17]:
def digital_root(N):
    return 0 if N == 0 else 1 + (N - 1) % 9

N = 12552031545
print(digital_root(N), get_lucky("leetcode", 10))   # 6 6 — trùng khi k đủ lớn
print(digital_root(9999), get_lucky("iiii", 1))     # 9 36 — KHÁC khi k nhỏ

6 6
9 36


## 9. Code nộp LeetCode (Python 3)

In [18]:
class Solution:
    def getLucky(self, s: str, k: int) -> int:
        num = 0
        for ch in s:
            p = ord(ch) - ord("a") + 1
            num += p // 10 + p % 10          # Transform #1 gộp với Convert
        for _ in range(k - 1):
            total = 0
            while num > 0:
                total += num % 10
                num //= 10
            num = total
        return num


sol = Solution()
assert sol.getLucky("iiii", 1) == 36
assert sol.getLucky("leetcode", 2) == 6
assert sol.getLucky("zbax", 2) == 8
print("3 ví dụ của đề PASS")

3 ví dụ của đề PASS


Bản ngắn, bám sát đề (cũng được chấp nhận vì Python có số nguyên lớn):

```python
class Solution:
    def getLucky(self, s: str, k: int) -> int:
        num = "".join(str(ord(c) - 96) for c in s)
        for _ in range(k):
            num = str(sum(int(d) for d in num))
        return int(num)
```

## 10. Vì sao thuật toán đúng?

**Mệnh đề:** tổng chữ số của chuỗi ghép `A + B` (nối chuỗi) bằng tổng chữ số của `A` cộng tổng chữ số của `B`. Lý do: tập các ký tự chữ số của `A + B` đúng bằng hợp (có lặp) các ký tự của `A` và `B`; phép cộng không phụ thuộc thứ tự.

Áp dụng lặp cho các mảnh `str(p₁), str(p₂), …` → Transform #1 = `Σ digit_sum(pᵢ)`. Với `1 ≤ p ≤ 26`, `digit_sum(p) = p // 10 + p % 10` (chữ số hàng chục + hàng đơn vị). Vậy **Đoạn 1** cho đúng giá trị sau Transform #1.

**Đoạn 2** thực hiện đúng `k - 1` lần Transform còn lại; vòng `while` tính `digit_sum` với invariant *`total + digit_sum(num_hiện_tại) = digit_sum(num_ban_đầu)`*: mỗi bước chuyển chữ số cuối từ `num` sang `total`. Khi `num = 0`, `total` là tổng chữ số.

Tổng cộng: 1 + (k − 1) = **k** lần Transform, đúng đề.

## 11. Time / Space Complexity

Gọi `n = len(s)`.

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| 1. Mô phỏng nguyên văn (`int` lớn) | `O(n)` cho dựng số + `O(n)` cho tổng chữ số đầu (thực tế `int(str)` lớn có chi phí cao hơn tuyến tính một chút) | `O(n)` | dễ viết, phụ thuộc số lớn |
| 2. Gộp Transform #1 (chính) | `O(n + k·log₁₀(1000))` = `O(n + k)` | `O(1)` | nên nộp |
| 3. Dừng sớm | `O(n + k)` | `O(1)` | tối ưu nhỏ |
| 4. Chuỗi chữ số | `O(n + k)` | `O(n)` | tránh số lớn, vẫn tốn chuỗi |

Sau Transform #1 giá trị ≤ 1000 (≤ 4 chữ số), nên mỗi lần Transform sau tốn hằng số bước.

## 12. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

In [19]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.getLucky("iiii", 1), 36)
check("đề 2", sol.getLucky("leetcode", 2), 6)
check("đề 3", sol.getLucky("zbax", 2), 8)
check("một chữ 'a', k=1", sol.getLucky("a", 1), 1)
check("chữ 'z' (26), k=1", sol.getLucky("z", 1), 8)
check("chữ 'j' (10), k=1", sol.getLucky("j", 1), 1)    # 10 -> 1 + 0 (chữ số 0 ở giữa)
check("k lớn hơn cần", sol.getLucky("leetcode", 10), 6)
check("100 chữ 's', k=1", sol.getLucky("s" * 100, 1), 1000)   # cận trên của Transform #1
check("100 chữ 's', k=2", sol.getLucky("s" * 100, 2), 1)

PASS đề 1: 36
PASS đề 2: 6
PASS đề 3: 8
PASS một chữ 'a', k=1: 1
PASS chữ 'z' (26), k=1: 8
PASS chữ 'j' (10), k=1: 1
PASS k lớn hơn cần: 6
PASS 100 chữ 's', k=1: 1000
PASS 100 chữ 's', k=2: 1


In [20]:
# Đối chiếu các cách trên dữ liệu ngẫu nhiên có seed
import random
random.seed(1945)
funcs = [get_lucky_literal, get_lucky, get_lucky_early_stop, get_lucky_string, Solution().getLucky]
for _ in range(500):
    s = "".join(random.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(random.randint(1, 100)))
    k = random.randint(1, 10)
    results = {f(s, k) for f in funcs}
    assert len(results) == 1, (s, k, results)
print("500 bộ ngẫu nhiên: mọi cách cho cùng kết quả")

500 bộ ngẫu nhiên: mọi cách cho cùng kết quả


## 13. Sai lầm người mới hay mắc

### Lỗi 1 — Dùng vị trí 0-based (`ord(ch) - ord('a')`)

`'a'` phải là **1**, không phải 0.

In [21]:
def buggy_zero_based(s, k):
    num = int("".join(str(ord(ch) - ord("a")) for ch in s))   # thiếu + 1
    for _ in range(k):
        num = sum(int(d) for d in str(num))
    return num

print("buggy:", buggy_zero_based("zbax", 2), "| đúng:", get_lucky("zbax", 2))

buggy: 4 | đúng: 8


### Lỗi 2 — Cộng **vị trí** thay vì cộng **chữ số** của vị trí

`sum(ord(ch) - 96 for ch in s)` cộng `26 + 2 + 1 + 24 = 53` — đó không phải Transform #1 (`17`). Ta phải cộng **từng chữ số**: `2+6+2+1+2+4`.

In [22]:
def buggy_sum_positions(s, k):
    num = sum(ord(ch) - 96 for ch in s)       # SAI: 53 thay vì 17
    for _ in range(k - 1):
        num = sum(int(d) for d in str(num))
    return num

print("buggy:", buggy_sum_positions("zbax", 2), "| đúng:", get_lucky("zbax", 2))

buggy: 8 | đúng: 8


Thú vị: bản lỗi trả `8` — **trùng** đáp án! Vì `53 → 8` và `17 → 8` cùng dư khi chia 9 (digital root). Test `"zbax", k=2` **không** bắt được lỗi này; hãy thử `k = 1`:

In [23]:
print("buggy k=1:", buggy_sum_positions("zbax", 1), "| đúng k=1:", get_lucky("zbax", 1))   # 53 vs 17

buggy k=1: 53 | đúng k=1: 17


Bài học: test với `k` nhỏ (k = 1) để thấy giá trị trung gian, đừng chỉ test ca mà mọi thứ "hội tụ" về cùng một chữ số.

### Lỗi 3 — Lặp `k` lần **sau khi** đã gộp Transform #1 (thừa một lần)

Nếu Đoạn 1 đã làm Transform #1 thì Đoạn 2 chỉ được chạy `k - 1` lần.

In [24]:
def buggy_extra_transform(s, k):
    num = sum(digit_sum(ord(ch) - 96) for ch in s)   # đã là Transform #1
    for _ in range(k):                               # SAI: phải là k - 1
        num = digit_sum(num)
    return num

print("buggy:", buggy_extra_transform("iiii", 1), "| đúng:", get_lucky("iiii", 1))   # 9 vs 36

buggy: 9 | đúng: 36


### Lỗi 4 — Dùng `/` thay vì `//`

`num / 10` cho số thực (`float`), vòng `while num > 0` sẽ chạy rất lâu với các giá trị như `0.0017…` và tổng sai. Luôn dùng `//` khi bóc chữ số.

### Lỗi 5 — Quên reset `total` cho mỗi lần Transform

Nếu khai báo `total = 0` **ngoài** vòng `for _ in range(k - 1)`, tổng lần sau bị cộng dồn từ lần trước.

In [25]:
print(17 / 10, 17 // 10)   # 1.7 1

1.7 1


## 14. Mẹo nhận dạng pattern

- "Thay chữ cái bằng vị trí trong bảng chữ" → `ord(ch) - ord('a') + 1`.
- "Tổng chữ số" → `while n: total += n % 10; n //= 10` (hoặc qua `str`). Đây là "nguyên liệu" của rất nhiều bài: 258 (Add Digits), 1281, 2180, 1837…
- Đề mô tả "làm theo các bước" → **Simulation**: viết đúng các bước trước, sau đó tìm bước nào có thể **gộp/rút gọn** (ở đây: gộp Convert + Transform #1).
- Khi thấy "tổng chữ số lặp lại nhiều lần" → nhớ tới bất biến **mod 9** (digital root) để kiểm tra chéo.

## 15. Biến thể

### 15.1. LeetCode 258 — Add Digits (lặp tới khi còn một chữ số)

In [26]:
def add_digits_loop(num):
    while num >= 10:
        num = digit_sum(num)
    return num

def add_digits_formula(num):
    return 0 if num == 0 else 1 + (num - 1) % 9

for x in [0, 5, 38, 99999, 12552031545]:
    print(x, add_digits_loop(x), add_digits_formula(x))

0 0 0
5 5 5
38 2 2
99999 9 9
12552031545 6 6


### 15.2. Bảng chữ đảo ngược (`a = 26`, …, `z = 1`)

Chỉ cần đổi công thức vị trí: `p = ord('z') - ord(ch) + 1`.

In [27]:
def get_lucky_reversed_alphabet(s, k):
    num = sum(digit_sum(ord("z") - ord(ch) + 1) for ch in s)
    for _ in range(k - 1):
        num = digit_sum(num)
    return num

print(get_lucky_reversed_alphabet("a", 1))    # a = 26 -> 8
print(get_lucky_reversed_alphabet("zbax", 1)) # z=1, b=25, a=26, x=3 -> 1+7+8+3 = 19

8
19


### 15.3. Đếm số lần Transform cần để về một chữ số

In [28]:
def steps_to_single_digit(s):
    num = sum(digit_sum(ord(ch) - 96) for ch in s)
    steps = 1
    while num >= 10:
        num = digit_sum(num)
        steps += 1
    return steps, num

print(steps_to_single_digit("leetcode"))   # (2, 6)
print(steps_to_single_digit("iiii"))       # (2, 9): 36 -> 9

(2, 6)
(2, 9)


## 16. Bài tập (TODO — tự làm, không chạy tự động)

**Bài 1 (dự đoán output):** `ord('j') - ord('a') + 1`, `divmod(19, 10)`, `sum(int(d) for d in "1000")` bằng bao nhiêu?

**Bài 2 (tính tay):** `s = "abc"`, `k = 1` và `s = "xyz"`, `k = 2`. Lập bảng chữ → vị trí → tổng chữ số.

**Bài 3 (điền chỗ trống):** hoàn thiện `exercise_digit_sum(n)` bằng `% 10` và `// 10`.

In [29]:
def exercise_digit_sum(n):
    total = 0
    while n > 0:
        # TODO 1: cộng chữ số cuối vào total
        # TODO 2: bỏ chữ số cuối của n
        pass
    return total

# assert exercise_digit_sum(262124) == 17
# assert exercise_digit_sum(0) == 0

**Bài 4 (tự viết lại):** viết `exercise_get_lucky(s, k)` theo cách gộp Transform #1, **không** tạo chuỗi chữ số dài.

In [30]:
def exercise_get_lucky(s, k):
    # TODO
    pass

# assert exercise_get_lucky("leetcode", 2) == 6
# assert exercise_get_lucky("zbax", 2) == 8

**Bài 5 (biến thể):** viết `exercise_add_digits(num)` (LeetCode 258) **không** dùng vòng lặp (dùng công thức mod 9).

**Bài 6 (mini challenge):** cho `k`, tìm chuỗi `s` gồm **đúng 3 chữ cái** cho kết quả `getLucky(s, 1)` **lớn nhất**; trả về giá trị lớn nhất đó. (Gợi ý: mỗi chữ cái đóng góp độc lập.)

In [31]:
def exercise_add_digits(num):
    # TODO
    pass

def exercise_max_lucky_three_letters():
    # TODO
    pass

# assert exercise_add_digits(38) == 2
# assert exercise_add_digits(0) == 0
# assert exercise_max_lucky_three_letters() == 30

## 17. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 1 | `j` là chữ thứ 10 | `divmod` trả (thương, dư) | `1+0+0+0` |
| 2 | đổi từng chữ | `x=24, y=25, z=26` | `(2+4)+(2+5)+(2+6) = 21 → 3` |
| 3 | lấy chữ số cuối | `n % 10` | `total += n % 10; n //= 10` |
| 4 | gộp Convert + Transform #1 | `p // 10 + p % 10` | sau đó lặp `k - 1` lần `digit_sum` |
| 5 | bất biến mod 9 | trường hợp `0` riêng | `0 if num == 0 else 1 + (num - 1) % 9` |
| 6 | chữ nào cho tổng chữ số lớn nhất? | đã tính ở mục 5 | `3 * max(digit_sum(p) for p in range(1, 27))` |

## 18. Đáp án tham khảo — chỉ xem sau khi tự làm

**Bài 1:** `10`, `(1, 9)`, `1`.

**Bài 2:** `"abc"` → `1 + 2 + 3 = 6`. `"xyz"` → `24, 25, 26` → `6 + 7 + 8 = 21` → Transform #2: `2 + 1 = 3`.

In [32]:
print(ord("j") - ord("a") + 1, divmod(19, 10), sum(int(d) for d in "1000"))
print(get_lucky("abc", 1), get_lucky("xyz", 2))


def answer_digit_sum(n):
    total = 0
    while n > 0:
        total += n % 10
        n //= 10
    return total


def answer_get_lucky(s, k):
    num = 0
    for ch in s:
        p = ord(ch) - ord("a") + 1
        num += p // 10 + p % 10
    for _ in range(k - 1):
        num = answer_digit_sum(num)
    return num


def answer_add_digits(num):
    return 0 if num == 0 else 1 + (num - 1) % 9


def answer_max_lucky_three_letters():
    return 3 * max(answer_digit_sum(p) for p in range(1, 27))


assert answer_digit_sum(262124) == 17
assert answer_digit_sum(0) == 0
assert answer_get_lucky("leetcode", 2) == 6
assert answer_get_lucky("zbax", 2) == 8
assert answer_add_digits(38) == 2
assert answer_add_digits(0) == 0
assert answer_max_lucky_three_letters() == 30
for x in range(0, 2000):
    assert answer_add_digits(x) == add_digits_loop(x)
print("Đáp án tham khảo PASS toàn bộ test!")

10 (1, 9) 1
6 3
Đáp án tham khảo PASS toàn bộ test!


## 19. Template nhớ nhanh + checklist submit

```python
class Solution:
    def getLucky(self, s: str, k: int) -> int:
        num = sum((ord(c) - 96) // 10 + (ord(c) - 96) % 10 for c in s)  # Convert + Transform #1
        for _ in range(k - 1):                                         # k - 1 lần còn lại
            num = sum(int(d) for d in str(num))
        return num
```

**Câu thần chú:** *chữ → vị trí 1..26 → cộng từng CHỮ SỐ → lặp đủ k lần (đã gộp lần đầu thì còn k − 1).*

**Checklist trước khi submit:**
- [ ] `'a'` là 1 (có `+ 1`)?
- [ ] Cộng **chữ số** của vị trí, không cộng vị trí?
- [ ] Đếm đúng tổng số lần Transform là `k`?
- [ ] Dùng `//` không dùng `/`; `total` reset mỗi lần?
- [ ] Đã test `k = 1` (để thấy giá trị trung gian) và chữ `'j'` (10, có chữ số 0)?